In [1]:
import time
import torch.backends.cudnn as cudnn
from torch import nn
from easydict import EasyDict as edict
from models import Generator, Discriminator, TruncatedVGG19
from datasets import SRDataset
from utils import *
from solver import train

In [2]:
# config
config = edict()
config.csv_folder = '..'
config.HR_data_folder = '../DIV2K_train_HR'
config.LR_data_folder = '../DIV2K_train_LR_bicubic/X4'
config.crop_size = 96 # HR图像裁减大小(W = crop_size, H = crop_size)
config.scaling_factor = 4 # n_pixel放大scaling_factor**2倍

# Generator parameters
config.G = edict()
config.G.large_kernel_size = 9
config.G.small_kernel_size = 3
config.G.n_channels = 64
config.G.n_blocks = 16
config.G.lr = 1e-4 # 生成器初始学习率
config.G.update_ratio = 2 # 生成器交替更新频率

# Discriminator parameters
config.D = edict()
config.D.kernel_size = 3
config.D.n_channels = 64
config.D.n_blocks = 8
config.D.fc_size = 1024
config.D.lr = 1e-5 # 判别器初始学习率
config.D.update_ratio = 1 # 判别器交替更新频率

# Learning parameters
config.checkpoint = None # path to model (SRGAN) checkpoint, None if none
config.batch_size = 16
config.start_epoch = 0
config.epochs = 40 # 论文要求先在lr=1e-4下训练1e+5iters，再在lr=1e-5下训练1e+5iters
config.workers = 4
config.vgg19_i = 5  # the index i in the definition for VGG loss; see paper
config.vgg19_j = 4  # the index j in the definition for VGG loss; see paper
config.beta = 1e-3  # the coefficient to weight the adversarial loss in the perceptual loss
config.print_freq = 500

# Default device
config.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cudnn.benchmark = True

In [3]:
if config.checkpoint is None:
    # Generator
    generator = Generator(config)

    # Initialize generator's optimizer
    optimizer_g = torch.optim.Adam(params=filter(lambda p: p.requires_grad, generator.parameters()),
                                   lr=config.G.lr)

    # Discriminator
    discriminator = Discriminator(config)
    optimizer_d = torch.optim.Adam(params=filter(lambda p: p.requires_grad, discriminator.parameters()),
                                   lr=config.D.lr)

else:
    checkpoint = torch.load(config.checkpoint)
    config.start_epoch = checkpoint['epoch'] + 1
    generator = checkpoint['generator']
    discriminator = checkpoint['discriminator']
    optimizer_g = checkpoint['optimizer_g']
    optimizer_d = checkpoint['optimizer_d']
    print("\nLoaded checkpoint from epoch %d.\n" % (checkpoint['epoch'] + 1))

In [4]:
# Truncated VGG19 network to be used in the loss calculation
truncated_vgg19 = TruncatedVGG19(i=config.vgg19_i, j=config.vgg19_j)
truncated_vgg19.eval()

/home/polaris-linux/miniconda3/envs/cv_env/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/polaris-linux/miniconda3/envs/cv_env/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG19_Weights.IMAGENET1K_V1`. You can also use `weights=VGG19_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


TruncatedVGG19(
  (truncated_vgg19): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): Conv2d(256, 256, kernel_size=(3, 3), s

In [5]:
# Loss functions
content_loss_criterion = nn.MSELoss()
adversarial_loss_criterion = nn.BCELoss() # 判别器最后一层为Sigmoid()，已经是概率，所以用BCELoss

In [6]:
# Move to default device
generator = generator.to(config.device)
discriminator = discriminator.to(config.device)
truncated_vgg19 = truncated_vgg19.to(config.device)
content_loss_criterion = content_loss_criterion.to(config.device)
adversarial_loss_criterion = adversarial_loss_criterion.to(config.device)

In [7]:
# Custom dataloaders
train_dataset = SRDataset(split='train', config=config)
train_loader = torch.utils.data.DataLoader(train_dataset,
                                           batch_size=config.batch_size,
                                           shuffle=True, 
                                           num_workers=config.workers,
                                           pin_memory=True)

In [8]:
# Epochs
for epoch in range(config.start_epoch, config.epochs):
    
    # At the halfway point, reduce learning rate to a tenth
    if epoch == config.epochs // 2:
        adjust_learning_rate(optimizer_g, 0.1)
        adjust_learning_rate(optimizer_d, 0.1)
    
    # One epoch's training
    train(train_loader=train_loader,
          generator=generator,
          discriminator=discriminator,
          truncated_vgg19=truncated_vgg19,
          update_ratio_g=config.G.update_ratio,
          update_ratio_d=config.D.update_ratio,
          content_loss_criterion=content_loss_criterion,
          adversarial_loss_criterion=adversarial_loss_criterion,
          optimizer_g=optimizer_g,
          optimizer_d=optimizer_d,
          epoch=epoch,
          device=config.device,
          beta=config.beta,
          print_freq=config.print_freq)
    # Save checkpoint
    torch.save({'epoch': epoch,
                'generator': generator,
                'discriminator': discriminator,
                'optimizer_g': optimizer_g,
                'optimizer_d': optimizer_d},
                '../checkpoint_srgan.pth.tar')

Epoch: [1][500/5000]----Batch Time 0.202 (0.230)----Data Time 0.000 (0.028)----Cont. Loss 0.5001 (0.3985)----Adv. Loss 4.3661 (2.6766)----Disc. Loss 1.0341 (0.5085)
Epoch: [1][1000/5000]----Batch Time 0.201 (0.240)----Data Time 0.000 (0.039)----Cont. Loss 0.4035 (0.4052)----Adv. Loss 2.9357 (2.6647)----Disc. Loss 0.2851 (0.5289)
Epoch: [1][1500/5000]----Batch Time 0.201 (0.259)----Data Time 0.000 (0.059)----Cont. Loss 0.4319 (0.4041)----Adv. Loss 1.7942 (2.9064)----Disc. Loss 0.3826 (0.4871)
Epoch: [1][2000/5000]----Batch Time 0.202 (0.268)----Data Time 0.000 (0.068)----Cont. Loss 0.3331 (0.4025)----Adv. Loss 4.5489 (3.2376)----Disc. Loss 0.4265 (0.4395)
Epoch: [1][2500/5000]----Batch Time 0.201 (0.274)----Data Time 0.000 (0.074)----Cont. Loss 0.2818 (0.4043)----Adv. Loss 4.9228 (3.4305)----Disc. Loss 0.1023 (0.4097)
Epoch: [1][3000/5000]----Batch Time 0.202 (0.278)----Data Time 0.000 (0.078)----Cont. Loss 0.6051 (0.4041)----Adv. Loss 3.8703 (3.6400)----Disc. Loss 0.0837 (0.3911)
Epoch